# MotionForge AI — geração SDXL pelo celular

Use este notebook pelo navegador do Android. O Colab fornece a GPU quando disponível; o telefone funciona como controle remoto. Os pesos e as imagens ficam no cache da sessão ou no Google Drive, nunca no Git.

SDXL é distribuído sob CreativeML Open RAIL++-M. Leia a licença e confirme que sua sessão possui GPU.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import sys, subprocess, torch
drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/MotionForgeData')
OUT = DATA / 'outputs'
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
print(DATA)


In [ ]:
!git clone --depth 1 --branch motionforge-multimodal-learning https://github.com/japaguimarassi-create/J.git /content/J
%cd /content/J
!python -m pip install -r generation/requirements.txt


In [ ]:
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Selecione uma GPU em Runtime > Change runtime type.')
print(torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


## Gerar uma imagem


In [ ]:
PROMPT = 'Original anime character concept art, dynamic pose, detailed cel shading, cinematic rim lighting'
NEGATIVE = 'blurry, low quality, malformed anatomy, duplicate limbs, text, watermark'
WIDTH, HEIGHT = 768, 768
STEPS = 25
SEED = 20261113
INIT_IMAGE = ''
LORA_DIR = DATA / 'sdxl-lora'
OUTPUT_IMAGE = OUT / 'motionforge-image.png'
cmd = [sys.executable, '-m', 'generation.image', '--prompt', PROMPT,
       '--negative-prompt', NEGATIVE, '--width', str(WIDTH), '--height', str(HEIGHT),
       '--steps', str(STEPS), '--seed', str(SEED), '--low-vram', '--output', str(OUTPUT_IMAGE)]
if INIT_IMAGE:
    cmd += ['--init-image', INIT_IMAGE, '--strength', '0.7']
if LORA_DIR.exists():
    cmd += ['--lora', str(LORA_DIR)]
subprocess.run(cmd, check=True)
from PIL import Image
display(Image.open(OUTPUT_IMAGE))
print('Saved:', OUTPUT_IMAGE)


## Opcional: enviar uma imagem inicial para transformar


In [ ]:
uploaded = files.upload()
for name, content in uploaded.items():
    target = DATA / name
    target.write_bytes(content)
    print('Set INIT_IMAGE =', str(target))


## Opcional: preparar um estilo próprio com LoRA

Coloque pairs.jsonl e as imagens correspondentes em MotionForgeData. O preparador aceita apenas CC0 1.0 e CC BY 4.0, verifica arquivos e remove duplicatas.


In [ ]:
PAIRS = DATA / 'pairs.jsonl'
TRAIN_DATA = DATA / 'sdxl_lora_dataset'
if PAIRS.is_file():
    subprocess.run([sys.executable, '-m', 'generation.prepare_sdxl_dataset',
                    '--pairs', str(PAIRS), '--output', str(TRAIN_DATA),
                    '--max-items', '3000', '--max-total-gb', '12'], check=True)
else:
    print('Coloque pairs.jsonl e os arquivos de mídia em MotionForgeData antes do treino.')


In [ ]:
!git clone --depth 1 https://github.com/huggingface/diffusers.git /content/J/third_party/diffusers
%cd /content/J
!python -m pip install -e third_party/diffusers
!python -m pip install -r generation/training-requirements.txt bitsandbytes


In [ ]:
LORA_OUT = DATA / 'sdxl-lora'
TRAIN_CMD = [sys.executable, '-m', 'generation.train_sdxl_lora',
             '--dataset-dir', str(TRAIN_DATA), '--diffusers-root', 'third_party/diffusers',
             '--steps', '1000', '--checkpoint-every', '200', '--batch-size', '1',
             '--gradient-accumulation', '4', '--rank', '16', '--resolution', '768',
             '--output', str(LORA_OUT), '--use-8bit-adam']
subprocess.run(TRAIN_CMD, check=True)
print('LoRA saved to Drive:', LORA_OUT)


## Retomar treinamento

Monte o mesmo Drive numa nova sessão, reinstale as dependências e execute o treino acrescentando --resume. O checkpoint precisa permanecer em MotionForgeData/sdxl-lora.
